## 01-摘要索引（Summary Index）

### 离线构建流程
1. 先将文档分块（块比较大），对每个块用 LLM 生成摘要；
2. 将摘要向量化，存入向量数据库；
3. 将摘要向量与原始文档快通过 ID 关联，原始文档块单独存储
4. 检索时，匹配摘要向量，返回原始文档快

### 注意事项：
- 摘要质量至关重要: 这是一个关于deepseek的介绍文章  太过于简单，丢失了关键信息
- 存储开销比较大:
    - 向量索引：只存储文档块对应的向量
    - 摘要索引：既要存储摘要的向量，还要存储原始文档
- 一致性问题，维护一致性: 通过 `uuid` 将原始文档和摘要一一对应

#### 对话模型

In [1]:
import os
from langchain_openai import ChatOpenAI
model = ChatOpenAI(api_key=os.getenv("DASHSCOPE_API_KEY"), model="qwen3.7-max", base_url="https://ws-ots0tb33v08803wb.cn-beijing.maas.aliyuncs.com/compatible-mode/v1")

#### 嵌入模型

In [3]:
from langchain_community.embeddings import DashScopeEmbeddings
embeddings_model = DashScopeEmbeddings(model="text-embedding-v4", dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"))

# 通过本地ollama上运行的bge-m3做向量化, 使用需要关闭 代理/VPN
from langchain_ollama import OllamaEmbeddings
embeddings_model = OllamaEmbeddings(model="bge-m3")

#### 文档加载

In [4]:
from langchain_community.document_loaders import TextLoader
# 加载数据
loader = TextLoader("../../../data/deepseek百度百科.txt", encoding="utf-8")
docs = loader.load()
print(f"文章的长度：{len(docs[0].page_content)}")

文章的长度：20899


#### 文档分块

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100)
docs = text_splitter.split_documents(docs)
print(f"文档块的数量：{len(docs)}")
print(f"第一个文档块的内容：{docs[0].page_content}")

文档块的数量：27


#### 用 LLM 为每个块生成摘要

In [6]:
from langchain_core.output_parsers import StrOutputParser   # 字符串输出解析器
from langchain_core.prompts import ChatPromptTemplate  # 聊天提示模板

# 创建摘要生成链
chain = (
    {"doc": lambda x: x.page_content} # doc.page_content
    | ChatPromptTemplate.from_template("总结下面的文档:\n\n{doc}")
    | model
    | StrOutputParser()
)

print("准备生成文档摘要，请耐心等待...")
summaries = chain.batch(docs, {"max_concurrency": 5}) # 批量生成文档摘要（最大并发数5）
print(f"摘要生成完成，共生成 {len(summaries)} 个摘要")

准备生成文档摘要，请耐心等待...
摘要生成完成，共生成 27 个摘要


In [7]:
print("-------------原始文档--------------")
print(docs[0].page_content)
print("-------------摘要--------------")
print(summaries[0])

-------------原始文档--------------
DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。
2024年1月5日至6月，相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2模型。 2024年9月5日，更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat，推出DeepSeek V2.5。12月13日，发布DeepSeek-VL2。12月26日，正式上线DeepSeek-V3首个版本并同步开源。2025年1月31日，英伟达宣布DeepSeek-R1模型登陆NVIDIANIM。同一时段内，亚马逊和微软也接入DeepSeek-R1模型。 2月5日，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。2月6日消息，澳大利亚政府以所谓“担心安全风险”为由，已禁止在所有政府设备中使用DeepSeek。2月8日，DeepSeek正式登陆苏州，并在苏州市公共算力服务平台上完成部署上线，为用户提供开箱即用的软硬件一体服务。 截至2月9日，DeepSeek App的累计下载量已超1.1亿次，周活跃用户规模最高近9700万。 2月20日消息，已有超200家企业宣布接入DeepSeek。 2月26日，DeepSeek宣布开源DeepGEMM。
-------------摘要--------------
以下是对该文档的结构化总结：

**核心摘要**
DeepSeek（深度求索）是由幻方量化孵化的人工智能创新企业，专注于大语言模型研发。凭借卓越的技术迭代和开源策略，其产品在全球范围内获得了庞大的用户基础和科技巨头的认可，但也面临部分海外政府的安全审查限制。

---

### 1. 公司背景与定位
* **基本信息**：全称杭州深度求索人工智能

#### 存储 + 索引构建
- 向量存储：Chroma, 存储摘要向量
- 文档存储：InMemoryByteStore, 存储原始文档
- 检索器：MultiVectorRetriever

In [8]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryByteStore
# 初始化Chroma实例（用于存储摘要向量）
vectorstore = Chroma(collection_name="summaries", embedding_function=embeddings_model)

# 初始化内存字节存储（用于存储原始文档）
store = InMemoryByteStore()

# 初始化多向量检索器（结合向量存储和文档存储）
id_key = "doc_id"
retriever = MultiVectorRetriever(
    vectorstore=vectorstore, # 向量检索库。存的是摘要向量，用来做相似度匹配
    byte_store=store, # 原始文档存储库。存的是原始文档chunk，通过ID关联
    id_key=id_key, # 关联键名。用于将摘要和原始文档进行关联，指定document中元数据中哪个字段作为"桥梁"，把向量库的结果映射回原文
)

将`摘要向量`添加到向量数据库（Chroma），将`原始文档块`存储到内存数据库（InMemoryByteStore）

In [9]:
import uuid   # 用于生成唯一标识符
from langchain_core.documents import Document    # 文档对象

# 为每个文档生成唯一ID，该ID用于关联原始文档和摘要
doc_ids = [str(uuid.uuid4()) for _ in docs]

# 将文档摘要转换为LangChain中Document
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# 将摘要向量添加到向量数据库
print("准备将摘要添加到向量数据库...")
batch_size = 5
for i in range(0, len(summary_docs), batch_size):
    batch = summary_docs[i:i + batch_size]
    retriever.vectorstore.add_documents(batch) # 核心功能：将摘要向量添加到向量数据库，batch_size最大是20，超过20会报错，模型要求
    print(f"已添加 {min(i + batch_size, len(summary_docs))}/{len(summary_docs)} 条")
print("摘要添加完成")
# 将原始文档存储到字节存储（使用ID关联）
print("准备将原始文档存储到字节存储...")
# mset：批量设置键值对
# list(zip(doc_ids, docs))：将ID和文档配对
retriever.docstore.mset(list(zip(doc_ids, docs)))
print("原始文档存储完成")

准备将摘要添加到向量数据库...
已添加 5/27 条
已添加 10/27 条
已添加 15/27 条
已添加 20/27 条
已添加 25/27 条
已添加 27/27 条
准备将原始文档存储到字节存储...


目前向量和文档存储都是存在本地

- 向量存储：生产环境不用Chroma，用Faiss或Milvus

- 文档存储：生产环境不用InMemoryByteStore，用MySQL或PG


数据关系图
```
vectorstore（Chroma）                    docstore（InMemoryByteStore）
┌────────────────────────┐              ┌────────────────────────┐
│ 记录1:                  │             │                        │
│   embedding: [0.02,..] │              │  "a1b2c3d4-..."        │
│   content: "摘要文本1"  │── doc_id ──→│    → docs[0] 原始文档块 │
│   metadata:            │   关联       │                        │
│     doc_id:"a1b2c3d4-" │              │  "e5f6g7h8-..."        │
├────────────────────────┤              │    → docs[1] 原始文档块 │
│ 记录2:                  │              │                        │
│   embedding: [0.15,..] │              │  "i9j0k1l2-..."        │
│   content: "摘要文本2"  │── doc_id ──→ │    → docs[2] 原始文档块 │
│   metadata:            │   关联       │                        │
│     doc_id:"e5f6g7h8-" │              │  ...                   │
├────────────────────────┤              │                        │
│ ...                    │              │                        │
└────────────────────────┘              └────────────────────────┘

```

检索流程中的调用顺序
```
    ① 搜索                    ② 取ID              ③ 取原文
Query ──────────→ vectorstore ──────────→ doc_id ──────────→ docstore ──→ 原始文档
                  (Chroma)                (桥梁)              (ByteStore)

                  做什么：                做什么：             做什么：
                  向量相似度匹配          从metadata提取       按key精确查找
                  返回Top-K摘要          doc_id字段          返回原始chunk

```

In [10]:
# 手动测试代码 - 相似性搜索
query = "deepseek的企业事件"
sub_docs = retriever.vectorstore.similarity_search(query)
print("-------------匹配的摘要内容--------------")
print(sub_docs[0])

# 获取第一个匹配摘要的ID
matched_id = sub_docs[0].metadata[id_key]

print(f"-------------对应的原始文档--------------")
# 通过ID获取原始文档
original_doc = retriever.docstore.mget([matched_id])
print(original_doc)
# 执行相似性搜索测试---完成

-------------匹配的摘要内容--------------
page_content='以下是对该文档的精炼总结：

**核心概况**
DeepSeek 是由量化巨头幻方量化创立的专注于大语言模型（LLM）研发的科技公司。依托母公司强大的算力支持，公司在成立后的一年多时间里，实现了核心模型的快速迭代与行业落地。

**关键发展历程**
*   **2023年7月（公司创立与算力奠基）**：7月17日正式成立。作为大厂外唯一储备万张A100芯片的公司，幻方量化为其提供了顶尖的硬件与算力支持。
*   **2023年8月（工商信息变更）**：8月2日完成工商调整，注册资本变更为1000万元，并变更了投资人及企业类型（其他有限责任公司）。
*   **2024年5月（V2模型发布与行业应用）**：5月7日发布 MoE 架构的 DeepSeek V2。该模型因架构和性能优异，被第一财经技术中心选为财经垂类大模型的主力基座模型并投入研发。
*   **2024年9月（V2.5模型升级）**：9月5日，官方合并了 Coder V2 和 V2 Chat 模型，推出全新的 DeepSeek V2.5，并保持 API 接口的向前兼容，方便用户无缝过渡。 

**总结论**
DeepSeek 的发展呈现出 **“高起点（顶级算力支持）+ 快迭代（短期内推出V2到V2.5）+ 强落地（获权威财经媒体技术认可）”** 的显著特征。' metadata={'doc_id': 'cb1edfb2-b399-4a4f-8e0e-81479ff7a945'}
-------------对应的原始文档--------------
[Document(metadata={'source': '../../data/deepseek百度百科.txt'}, page_content='发展历程\n早期发展\nDeepSeek成立于2023年7月17日，由知名量化资管巨头幻方量化创立。DeepSeek是一家创新型科技公司，长久以来专注于开发先进的大语言模型（LLM）和相关技术，作为大厂外唯一一家储备万张A100芯片的公司，幻方量化为DeepSeek的技术研发提供了强大的硬件支持。\n2023年8月2日，注册资本变更为1000万元，章程备案，投资人变更为宁波程恩企业管理咨询合伙企业（有限合伙），市场

In [11]:
# retriever.invoke将上面对摘要进行检索，但是通过关联ID获得原始文档，最终返回原始文档的过程全部都包含完成了
retriever.invoke("deepseek的企业事件")[0]

Document(metadata={'source': '../../data/deepseek百度百科.txt'}, page_content='发展历程\n早期发展\nDeepSeek成立于2023年7月17日，由知名量化资管巨头幻方量化创立。DeepSeek是一家创新型科技公司，长久以来专注于开发先进的大语言模型（LLM）和相关技术，作为大厂外唯一一家储备万张A100芯片的公司，幻方量化为DeepSeek的技术研发提供了强大的硬件支持。\n2023年8月2日，注册资本变更为1000万元，章程备案，投资人变更为宁波程恩企业管理咨询合伙企业（有限合伙），市场主体类型变更为其他有限责任公司。\n2024年5月7日，DeepSeek发布了MoE架构的DeepSeek V2。两天后，第一财经技术中心就对DeepSeek发布的技术报告和模型进行了深度分析与研究。经过严谨的评估，团队认为DeepSeek V2在架构设计、性能表现等方面与财经垂类大模型的需求高度契合，于是果断决定将基座模型迁移至DeepSeek V2，并迅速启动了以DeepSeek V2为主力基座模型的财经垂类大模型的训练与应用研发工作。\n2024年9月5日，DeepSeek官方更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat两个模型，升级推出全新的DeepSeek V2.5新模型。官方表示为向前兼容，API用户通过deepseek-coder或deepseek-chat均可以访问新的模型。')

### 4. 问题回答

增强 + 生成

In [12]:
from langchain_core.output_parsers import StrOutputParser   # 字符串输出解析器
from langchain_core.prompts import ChatPromptTemplate  # 聊天提示模板
from langchain_core.runnables import RunnableMap, RunnableParallel  # 可运行的映射
prompt =  ChatPromptTemplate.from_template("根据下面的文档回答问题:\n\n{doc}\n\n问题: {question}")
# 生成问题回答链
# RunnableMap是RunnableParallel另一种写法
# retriever.invoke将上面对摘要进行检索，但是通过关联ID获得原始文档，最终返回原始文档的过程全部都包含完成了
chain = RunnableMap({
    "doc": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | model | StrOutputParser()

# 生成问题回答
query = "deepseek的企业事件"
answer = chain.invoke({"question": query})
print("-------------回答--------------")
print(answer)
# retriever.invoke(query)  1.向量数据库中检索摘要向量   2.匹配对应的原始文档并返回
retrieved_docs = retriever.invoke(query)
print("-------------检索到的文档--------------")
print(retrieved_docs)

-------------回答--------------
根据提供的文档，DeepSeek（杭州深度求索人工智能基础技术研究有限公司）的企业事件可以按照以下几个主要方面进行梳理和总结：

### 一、 公司成立与早期发展
* **2023年7月17日**：DeepSeek正式成立，由知名量化资管巨头幻方量化创立，专注于大语言模型（LLM）及相关技术开发。幻方量化为其提供了强大的硬件支持（储备万张A100芯片）。
* **2023年8月2日**：公司注册资本变更为1000万元，投资人变更为宁波程恩企业管理咨询合伙企业（有限合伙），市场主体类型变更为其他有限责任公司。

### 二、 模型发布与技术开源
* **2024年1月至6月**：相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2（MoE架构）、DeepSeek-Coder-V2等模型。
* **2024年9月5日**：更新API支持文档，合并DeepSeek Coder V2和DeepSeek V2 Chat，推出全新的DeepSeek V2.5模型。
* **2024年12月13日**：发布DeepSeek-VL2。
* **2024年12月26日**：正式上线DeepSeek-V3首个版本并同步开源。
* **2025年2月26日**：宣布开源DeepGEMM。

### 三、 企业合作与平台接入
* **2024年5月**：第一财经技术中心经评估后，果断将基座模型迁移至DeepSeek V2，启动财经垂类大模型研发。
* **2025年1月31日**：英伟达宣布DeepSeek-R1模型登陆NVIDIA NIM；同一时段内，亚马逊和微软也接入DeepSeek-R1模型。
* **2025年2月**：腾讯云「HAI」支持DeepSeek-R1一键部署（开发者仅需3分钟接入）；居然智家数智化业务全线接入（含V3和R1版本）并启动私有化部署。
* **2025年2月11日**：百度文心一言接入DeepSeek；极狐汽车（阿尔法S5）深度融合DeepSeek大模型实现智能座舱进化；若羽臣经由阿里云正式接入。
* **2025年2月12日**：马蜂窝自研AI应用、猿辅导集团（小猿学练机、小猿口算等）、顺丰同城相继接入；景嘉微宣布